# 📓 Semana 8 · Dia 6 — Entregável: SCD2 + CDC + tuning + simulado DEP

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEP (simulado) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | SCD2 + CDC + benchmark + simulado ≥ 70% |

---


## 📖 Teoria — O que você agora sabe fazer

- Tuning: plano físico, broadcast, AQE, cache (10x)
- Organização física: Liquid Clustering vs particionamento
- CDC: Change Data Feed (ler mudanças)
- SCD1/SCD2: APPLY CHANGES INTO
- DLT avançado: triggered/continuous + expectations combinadas

Esse é o núcleo da **Data Engineer Professional**.


### 💻 Na prática — Entregável integrado

Monte o pipeline completo: Bronze (CDF on) → Prata SCD2 → Ouro com tuning.


In [ ]:
# Passo 1: garantir CDF no Bronze (para CDC)
spark.sql("ALTER TABLE workspace.bronze.vendas_bronze SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print("CDF habilitado no Bronze.")

In [ ]:
# Passo 2: benchmark antes do tuning
import time
t0 = time.time()
r = spark.sql("""
  SELECT Country, DATE_TRUNC('month', InvoiceDate) mes, SUM(Quantity*UnitPrice) receita
  FROM workspace.bronze.vendas_bronze GROUP BY 1, 2
""").count()
t1 = time.time()
print(f"Antes do tuning: {t1-t0:.2f}s ({r} linhas)")

In [ ]:
# Passo 3: aplicar tuning (broadcast em dimensão pequena)
dim = spark.table("workspace.prata.dim_produto").limit(5)
t0 = time.time()
r2 = (spark.table("workspace.prata.fato_vendas")
    .join(dim.hint("broadcast"), "sk_produto", "left")
    .groupBy("Country").count().count())
t1 = time.time()
print(f"Com broadcast: {t1-t0:.2f}s")

### 💻 Na prática — Simulado DEP parcial (10 questões)

Marque antes do gabarito.


### Questões

**1.** O AQE pode:
- A) transformar sort-merge em broadcast em runtime  B) apagar dados
- C) criar tabelas  D) nada

**2.** Para leitura eficiente por país+data em tabela de 5 TB:
- A) particionar por país  B) Liquid Clustering (Country, data)  C) Z-ORDER só  D) nada

**3.** `_change_type='delete'` aparece ao ler com:
- A) readChangeFeed=true  B) readStream normal  C) select simples  D) count

**4.** SCD2 com histórico é feito com:
- A) MERGE simples  B) apply_changes (stored_as_scd_type=2)  C) UPDATE  D) INSERT OVERWRITE

**5.** `sequence_by` no apply_changes define:
- A) ordem das colunas  B) ordem temporal dos eventos  C) tamanho da tabela  D) nada

**6.** Continuous vs Triggered: Continuous é:
- A) mais barato  B) streaming contínuo (baixa latência)  C) batch único  D) igual

**7.** `expect_all_or_drop`:
- A) mantém e conta  B) descarta violadas  C) falha pipeline  D) remove a tabela

**8.** Para propagar mudanças do Delta a outro sistema:
- A) CDF  B) cache  C) broadcast  D) VACUUM

**9.** VACUUM com retenção padrão (7d):
- A) apaga tudo  B) remove arquivos fora da retenção  C) nunca roda  D) destrói o log

**10.** Spark UI mostra:
- A) DAG, estágios, tasks  B) apenas erros  C) nada  D) somente SQL


## 📖 Teoria — Gabarito

**1-A** · **2-B** · **3-A** · **4-B** · **5-B** · **6-B** · **7-B** · **8-A** · **9-B** · **10-A**. ≥ 7/10 = pronto para a Semana 9.


> 🎯 **Dica de prova**: DEP é a prova de 'para que serve X'. Ao responder, pergunte-se: o que essa ferramenta RESOLVE? (CDF=propagar mudança, AQE=otimizar runtime, CLUSTER BY=pruning em cardinalidade alta).


## 🎯 Exercícios de fixação

**1.** Documente o benchmark antes/depois do seu tuning no README.

**2.** Explique o fluxo Bronze(CDF) → Prata(SCD2) → Ouro em 4 frases.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Benchmark

Anote tempo, nº de arquivos e a técnica (broadcast/cache/cluster). É a prova de impacto para entrevistas.

**2.** Fluxo

Bronze captura mudanças (CDF); Prata mantém dimensões SCD2 (histórico); Ouro agrega para BI; o DLT orquestra tudo com expectations e triggered.



## ✅ Checklist de fechamento

- [ ] Apliquei broadcast/AQE/cache e documentei o benchmark.
- [ ] Habilitei CDF e li mudanças.
- [ ] Implementei SCD2 (demo + APPLY CHANGES).
- [ ] Fiz o simulado DEP parcial e revisei erros.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*